In [1]:
# ==========================================
# KNN - KLASIFIKASI GENRE MUSIK SPOTIFY
# Dataset: Spotify 1500 Data
# ==========================================

# 1. Import library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [5]:
# DATA LOADING
# Membaca dataset Spotify
df = pd.read_csv("spotify.csv")

print("Ukuran dataset (baris, kolom):", df.shape)
display(df.head())
print("\nNama kolom:")
print(df.columns.tolist())

Ukuran dataset (baris, kolom): (1500, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,51662,78PDj0uocOvtSJCdOt65qy,Yo Yo Honey Singh,Bhaag Johnny,Aankhon Aankhon,59,244831,False,0.746,0.988,...,-4.565,0,0.0637,0.2900,0.002180,0.131,0.610,130.042,4,hip-hop
1,81761,1ei3hzQmrgealgRKFxIcWn,Benson Boone,In The Stars,In The Stars,86,216410,False,0.362,0.540,...,-6.031,1,0.0528,0.3380,0.000000,0.139,0.300,78.007,3,pop
2,16713,38znqxzVreO8WnoEkBIAJI,Wolfgang Amadeus Mozart;Academy of St. Martin ...,Mozart - Inspiring Classics,"Symphony No. 22 in C, K.162: 3. Presto assai",1,98506,False,0.494,0.216,...,-17.779,1,0.0452,0.9660,0.921000,0.115,0.641,76.432,3,classical
3,16816,2baiU53SqqdVf1Kdlbmc06,Wolfgang Amadeus Mozart;Quartetto Italiano,Mozart - A Classical Dawn,"String Quartet No. 9 in A, K.169: 4. Rondeaux ...",3,98160,False,0.470,0.124,...,-18.936,1,0.0522,0.9380,0.000076,0.198,0.911,117.189,4,classical
4,19737,5jQxNgMrDlbuYO6LQ3aY4Y,Lasse Stefanz,Greatest Hits,Du försvann som en vind,47,208120,False,0.633,0.696,...,-6.626,1,0.0296,0.0914,0.000041,0.266,0.714,112.992,4,country



Nama kolom:
['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


In [9]:
# DATA CLEANING
# Merapikan nama kolom
df.columns = df.columns.str.strip().str.lower()

In [22]:
# Menentukan fitur audio
fitur = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

In [23]:
# Menentukan target yang ingin diprediksi
target = "track_genre"

In [24]:
# Memeriksa apakah kolom yang dibutuhkan tersedia
kolom_wajib = fitur + [target]

kolom_tidak_ada = [
    kolom for kolom in kolom_wajib
    if kolom not in df.columns
]

if kolom_tidak_ada:
    raise ValueError(
        "Kolom tidak ditemukan: "
        + ", ".join(kolom_tidak_ada)
    )

In [25]:
# Mengubah fitur audio menjadi angka
for kolom in fitur:
    df[kolom] = pd.to_numeric(
        df[kolom],
        errors="coerce"
    )

In [26]:
# Mengecek data kosong sebelum dibersihkan
print("Data kosong sebelum cleaning:")
display(df[kolom_wajib].isna().sum().to_frame("Jumlah Kosong"))

Data kosong sebelum cleaning:


,Jumlah Kosong
danceability,0
energy,0
loudness,0
speechiness,0
acousticness,0
instrumentalness,0
liveness,0
valence,0
tempo,0
track_genre,0


In [27]:
# Mengecek jumlah data duplikat
print("Jumlah data duplikat:", df.duplicated().sum())


Jumlah data duplikat: 0


In [28]:
# Menghapus data duplikat
jumlah_awal = len(df)
df = df.drop_duplicates()

In [29]:
# Menghapus baris yang kosong pada kolom yang digunakan
df = df.dropna(subset=kolom_wajib)


In [30]:
# Membersihkan nama genre
df[target] = df[target].astype(str).str.strip()
df = df[df[target] != ""]


In [31]:
# Menampilkan hasil cleaning
print("Jumlah baris sebelum cleaning:", jumlah_awal)
print("Jumlah baris setelah cleaning:", len(df))
print("Ukuran dataset setelah cleaning:", df.shape)

print("Jumlah lagu berdasarkan genre:")
display(df[target].value_counts().to_frame("Jumlah Lagu"))

if df.empty:
    raise ValueError(
        "Dataset kosong setelah cleaning. "
        "Periksa dataset Spotify."
    )

Jumlah baris sebelum cleaning: 1500
Jumlah baris setelah cleaning: 1500
Ukuran dataset setelah cleaning: (1500, 21)
Jumlah lagu berdasarkan genre:


,Jumlah Lagu
track_genre,
hip-hop,150
pop,150
classical,150
country,150
disco,150
dance,150
jazz,150
blues,150
acoustic,150


In [51]:
# DATA SPLITTING6
# Memisahkan fitur dan target
X = df[fitur].copy()
y_text = df[target].copy()

In [52]:
# Mengecek jumlah data pada setiap genre
jumlah_per_genre = y_text.value_counts()


In [53]:
# Genre harus memiliki minimal 2 data
genre_valid = jumlah_per_genre[
    jumlah_per_genre >= 2
].index

mask = y_text.isin(genre_valid)

X = X.loc[mask]
y_text = y_text.loc[mask]

In [57]:
# Memastikan terdapat minimal 2 genre
if y_text.nunique() < 2:
    raise ValueError(
        "Klasifikasi membutuhkan minimal 2 genre, "
        "masing-masing dengan minimal 2 data."
    )

In [55]:
# Mengubah nama genre menjadi angka
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

In [56]:
# Membagi data training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [40]:
# Menampilkan hasil pembagian data
print("Jumlah data training:", X_train.shape[0])
print("Jumlah data testing:", X_test.shape[0])
print("Jumlah fitur:", X_train.shape[1])
print("Daftar genre:", list(label_encoder.classes_))

Jumlah data training: 1200
Jumlah data testing: 300
Jumlah fitur: 9
Daftar genre: ['acoustic', 'blues', 'classical', 'country', 'dance', 'disco', 'electronic', 'hip-hop', 'jazz', 'pop']


In [41]:
# MODELING KNN
# Normalisasi fitur menggunakan StandardScaler
scaler = StandardScaler()

# Scaler belajar dari data training
X_train_scaled = scaler.fit_transform(X_train)

# Data testing menggunakan skala yang sama
X_test_scaled = scaler.transform(X_test)

# Menentukan nilai K
k = min(5, len(X_train_scaled))

# Membuat model KNN
model_knn = KNeighborsClassifier(
    n_neighbors=k
)

# Melatih model menggunakan data training
model_knn.fit(
    X_train_scaled,
    y_train
)

print("Model KNN berhasil dilatih.")
print("Nilai K:", k)

Model KNN berhasil dilatih.
Nilai K: 5


In [50]:
# EVALUATION
# Memprediksi genre dari data testing
y_pred = model_knn.predict(X_test_scaled)

# Menghitung akurasi
akurasi = accuracy_score(y_test, y_pred)

print("Hasil Evaluasi Model KNN")
print("========================")
print(f"Akurasi: {akurasi:.4f}")
print(f"Akurasi dalam persen: {akurasi * 100:.2f}%")

# Membuat confusion matrix
label_angka = np.arange(len(label_encoder.classes_))

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=label_angka
)

print("\nConfusion Matrix:")

display(
    pd.DataFrame(
        cm,
        index=[
            f"Aktual: {g}"
            for g in label_encoder.classes_
        ],
        columns=[
            f"Prediksi: {g}"
            for g in label_encoder.classes_
        ]
    )
)

# Menampilkan precision, recall, dan F1-score
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        labels=label_angka,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Hasil Evaluasi Model KNN
Akurasi: 0.3967
Akurasi dalam persen: 39.67%

Confusion Matrix:


,Prediksi: acoustic,Prediksi: blues,Prediksi: classical,Prediksi: country,Prediksi: dance,Prediksi: disco,Prediksi: electronic,Prediksi: hip-hop,Prediksi: jazz,Prediksi: pop
Aktual: acoustic,13,0,1,6,1,2,1,0,5,1
Aktual: blues,5,8,0,3,3,1,0,1,4,5
Aktual: classical,2,1,24,1,0,0,0,0,2,0
Aktual: country,8,5,0,10,1,2,0,0,0,4
Aktual: dance,2,2,0,3,13,5,3,1,0,1
Aktual: disco,4,2,0,0,5,12,4,2,0,1
Aktual: electronic,1,5,0,0,5,3,10,3,2,1
Aktual: hip-hop,2,4,1,2,2,0,5,13,0,1
Aktual: jazz,6,1,0,7,1,0,0,1,14,0
Aktual: pop,6,1,0,3,6,1,1,6,4,2



Classification Report:
              precision    recall  f1-score   support

    acoustic       0.27      0.43      0.33        30
       blues       0.28      0.27      0.27        30
   classical       0.92      0.80      0.86        30
     country       0.29      0.33      0.31        30
       dance       0.35      0.43      0.39        30
       disco       0.46      0.40      0.43        30
  electronic       0.42      0.33      0.37        30
     hip-hop       0.48      0.43      0.46        30
        jazz       0.45      0.47      0.46        30
         pop       0.12      0.07      0.09        30

    accuracy                           0.40       300
   macro avg       0.40      0.40      0.40       300
weighted avg       0.40      0.40      0.40       300

